# 1. Learned fusion routing and validation-tuned constant baseline

This is a **read-only, no-retraining follow-up** to the completed controlled learned-gate and fixed-equal-fusion experiments.

It answers the viva-facing questions:

1. Did the learned gate perform participant-specific routing, or mostly learn a run-specific constant?
2. Does the gate assign more interaction pathway weight when interaction pathway is the more reliable pathway?
3. Was fixed 50/50 an unfair baseline because another constant evidence weight would work better?
4. Are interaction pathway and evidence pathway sufficiently complementary for participant-level routing to have useful headroom?

For each seed/fold, a single constant interaction-pathway evidence weight is selected **only from that run's validation predictions**. The selected weight is frozen before application to the untouched test fold. Test labels never influence weight selection.

The native final fusion is reconstructed from the saved branch opinions and checkpoint evidence-scale parameters before analysis. The notebook refuses to continue unless reconstructed probabilities and uncertainties reproduce the saved native outputs within tolerance.


## 1.1. Exact protocol and isolated output directory

The complete controlled matrix is seeds 17, 42 and 73 across outer folds 0-4. The grid is fixed in advance at 101 weights from 0.00 to 1.00. Validation ROC AUC is the selection criterion; ties are resolved by validation negative log-likelihood, then proximity to 0.50, then the smaller weight.

All completed experiment folders are treated as read-only. New tables, predictions, figures and the final completion manifest are written only under a new comparison directory.


In [ ]:
# ============================================================
# 1. Exact paths, analysis settings, imports, and atomic outputs
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)


try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass


PROJECT_ROOT = Path("/content/drive/MyDrive/adni_mri")
MODEL_ROOT = PROJECT_ROOT / "models" / "3mt_tmc_evidential"

TASK_NAME = "mci_prognosis"
SEEDS = (17, 42, 73)
OUTER_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_PARTICIPANTS_PER_SEED = 544
CLASSIFICATION_THRESHOLD = 0.50
NUMBER_OF_CLASSES = 2

LEARNED_EXPERIMENT_NAME = "gated_cmt_learned_gate_md050_seed_sensitivity"
FIXED_EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_md050_seed_sensitivity"

LEARNED_TASK_ROOT = (
    MODEL_ROOT / "experiments" / LEARNED_EXPERIMENT_NAME / TASK_NAME
)
FIXED_TASK_ROOT = (
    MODEL_ROOT / "experiments" / FIXED_EXPERIMENT_NAME / TASK_NAME
)

COMPARISON_NAME = (
    "controlled_seed_validation_tuned_constant_vs_learned_gate_md050"
)
COMPARISON_ROOT = (
    MODEL_ROOT / "experiment_comparisons" / COMPARISON_NAME
)
TABLE_DIR = COMPARISON_ROOT / "tables"
PREDICTION_DIR = COMPARISON_ROOT / "paired_predictions"
FIGURE_DIR = COMPARISON_ROOT / "figures"

for directory in [COMPARISON_ROOT, TABLE_DIR, PREDICTION_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# Pre-specified validation-only search grid and deterministic tie rules.
ALPHA_GRID = np.round(np.linspace(0.0, 1.0, 101), 2)
PRIMARY_SELECTION_METRIC = "validation ROC AUC"
BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_BASE_SEED = 20260815
RECONSTRUCTION_TOLERANCE = 5e-5


BRANCH_MASK_COLUMNS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
    "BRANCH_MASK__MRI",
]
BRANCH_DISPLAY_NAMES = {
    "BRANCH_MASK__DEMOGRAPHICS": "Demographics",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL": "Cognitive/Functional",
    "BRANCH_MASK__CSF": "CSF",
    "BRANCH_MASK__PLASMA": "Plasma",
    "BRANCH_MASK__APOE": "APOE",
    "BRANCH_MASK__MRI": "MRI",
}


def learned_run_root(seed, fold):
    return LEARNED_TASK_ROOT / f"seed_{seed}" / f"fold_{fold}"


def fixed_run_root(seed, fold):
    return FIXED_TASK_ROOT / f"seed_{seed}" / f"fold_{fold}"


def atomic_write_csv(dataframe, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(
        f".{destination.stem}.tmp.{os.getpid()}{destination.suffix}"
    )
    dataframe.to_csv(temporary, index=False)
    os.replace(temporary, destination)


def atomic_write_json(value, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(
        f".{destination.name}.tmp.{os.getpid()}"
    )
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, destination)


def atomic_save_figure(figure, destination, dpi=180):
    destination = Path(destination)
    temporary = destination.with_name(
        f".{destination.stem}.tmp.{os.getpid()}{destination.suffix}"
    )
    figure.savefig(temporary, dpi=dpi, bbox_inches="tight")
    os.replace(temporary, destination)


def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print("=" * 72)
print("CONTROLLED GATE-ROUTING AND VALIDATION-TUNED CONSTANT FUSION")
print("=" * 72)
print("Learned runs:", LEARNED_TASK_ROOT)
print("Fixed runs:", FIXED_TASK_ROOT)
print("New comparison outputs:", COMPARISON_ROOT)
print("Weight grid:", f"{ALPHA_GRID[0]:.2f} to {ALPHA_GRID[-1]:.2f}",
      f"({len(ALPHA_GRID)} values)")
print("Selection criterion:", PRIMARY_SELECTION_METRIC)


## 1.2. Validate the 30 completed runs and reconstruct evidential fusion exactly

The code checks completion markers, file hashes, seed/fold identities, split hashes, architecture/training invariants and learned/fixed participant alignment. It also extracts the two learned evidence-scale parameters directly from each best checkpoint.

For a binary Dirichlet opinion with class probability (p_k) and uncertainty (u), strength is (S=2/u) and evidence is (e_k=p_kS-1). After applying the checkpoint scales, the same evidence mixture used by the trained model can be reconstructed for any constant weight.


In [ ]:
# ============================================================
# 2. Strict loading, identity checks, and exact reconstruction
# ============================================================

REQUIRED_PREDICTION_COLUMNS = {
    "RID",
    "TARGET",
    "FINAL_P_pMCI",
    "FINAL_UNCERTAINTY",
    "W_3MT",
    "W_TMC",
    "THREE_MT_P_pMCI",
    "THREE_MT_UNCERTAINTY",
    "TMC_P_pMCI",
    "TMC_UNCERTAINTY",
    "ORIGINAL_MODALITY_COUNT",
    "EFFECTIVE_MODALITY_COUNT",
}

TOP_LEVEL_IDENTITY_INVARIANTS = [
    "task",
    "fold",
    "random_seed",
    "schema_path",
    "schema_sha256",
    "input_path",
    "input_sha256",
    "train_rows_sha256",
    "validation_rows_sha256",
    "test_rows_sha256",
]

EXPECTED_GATE_ABLATION_DIFFERENCES = {
    "trainable_parameters",
    "gate_regularisation_epochs",
    "gate_loss_weight",
    "hybrid_fusion_class",
    "fusion_rule",
    "fixed_three_mt_weight",
    "fixed_tmc_weight",
}


def run_paths(root):
    return {
        "configuration": root / "history" / "training_configuration.json",
        "best_checkpoint": root / "checkpoints" / "best_validation_auc_checkpoint.pt",
        "validation": root / "predictions" / "best_validation_predictions.csv",
        "test": root / "predictions" / "test_predictions.csv",
        "metrics": root / "predictions" / "test_metrics.json",
        "success": root / "_SUCCESS.json",
    }


def load_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def stable_softplus(value):
    value = float(value)
    return float(max(value, 0.0) + math.log1p(math.exp(-abs(value))))


def load_checkpoint_scales(checkpoint_path):
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    state = checkpoint.get("model_state_dict")
    if not isinstance(state, dict):
        raise ValueError(f"Checkpoint has no model_state_dict: {checkpoint_path}")

    suffixes = {
        "THREE_MT_SCALE": "hybrid_fusion.three_mt_scale_parameter",
        "TMC_SCALE": "hybrid_fusion.tmc_scale_parameter",
    }
    result = {}
    for output_name, suffix in suffixes.items():
        matches = [key for key in state if key.endswith(suffix)]
        if len(matches) != 1:
            raise KeyError(
                f"Expected one checkpoint key ending in {suffix}; found {matches}."
            )
        raw = float(state[matches[0]].detach().cpu().reshape(-1)[0].item())
        result[output_name] = stable_softplus(raw)
        result[output_name + "_PARAMETER"] = raw
        result[output_name + "_KEY"] = matches[0]
    return result


def validate_prediction_table(table, label):
    missing = REQUIRED_PREDICTION_COLUMNS.difference(table.columns)
    if missing:
        raise KeyError(f"{label} lacks columns: {sorted(missing)}")
    if table["RID"].duplicated().any():
        raise ValueError(f"{label} contains duplicated RIDs.")
    if set(pd.to_numeric(table["TARGET"], errors="raise").astype(int)) - {0, 1}:
        raise ValueError(f"{label} has targets outside 0/1.")
    bounded = [
        "FINAL_P_pMCI", "FINAL_UNCERTAINTY", "W_3MT", "W_TMC",
        "THREE_MT_P_pMCI", "THREE_MT_UNCERTAINTY",
        "TMC_P_pMCI", "TMC_UNCERTAINTY",
    ]
    for column in bounded:
        numeric = pd.to_numeric(table[column], errors="raise")
        if not numeric.between(0.0, 1.0, inclusive="both").all():
            raise ValueError(f"{label}: {column} is outside [0, 1].")
    if not np.allclose(table["W_3MT"] + table["W_TMC"], 1.0, atol=1e-6):
        raise ValueError(f"{label}: weights do not sum to one.")


def opinion_to_evidence(positive_probability, uncertainty, label):
    probability = np.asarray(positive_probability, dtype=float)
    uncertainty = np.asarray(uncertainty, dtype=float)
    if np.any(~np.isfinite(probability)) or np.any(~np.isfinite(uncertainty)):
        raise ValueError(f"{label}: non-finite opinion values.")
    if np.any(uncertainty <= 0.0) or np.any(uncertainty > 1.0 + 1e-8):
        raise ValueError(f"{label}: uncertainty must lie in (0, 1].")

    strength = NUMBER_OF_CLASSES / uncertainty
    positive_evidence = probability * strength - 1.0
    negative_evidence = (1.0 - probability) * strength - 1.0
    evidence = np.column_stack([negative_evidence, positive_evidence])
    minimum = float(np.min(evidence))
    if minimum < -5e-4:
        raise ValueError(
            f"{label}: reconstructed evidence is materially negative ({minimum})."
        )
    return np.clip(evidence, 0.0, None)


def calibrated_branch_evidence(table, scales, label):
    three_mt = opinion_to_evidence(
        table["THREE_MT_P_pMCI"],
        table["THREE_MT_UNCERTAINTY"],
        label + " 3MT",
    ) * scales["THREE_MT_SCALE"]
    tmc = opinion_to_evidence(
        table["TMC_P_pMCI"],
        table["TMC_UNCERTAINTY"],
        label + " TMC",
    ) * scales["TMC_SCALE"]
    return three_mt, tmc


def mix_calibrated_evidence(three_mt_evidence, tmc_evidence, three_mt_weight):
    weight = np.asarray(three_mt_weight, dtype=float)
    if weight.ndim == 0:
        weight = np.full(len(three_mt_evidence), float(weight))
    weight = weight.reshape(-1, 1)
    if len(weight) != len(three_mt_evidence):
        raise ValueError("Weight vector length does not match evidence rows.")
    if np.any(weight < 0.0) or np.any(weight > 1.0):
        raise ValueError("Fusion weights must lie in [0, 1].")

    final_evidence = (
        weight * three_mt_evidence
        + (1.0 - weight) * tmc_evidence
    )
    final_alpha = final_evidence + 1.0
    strength = final_alpha.sum(axis=1)
    probability = final_alpha[:, 1] / strength
    uncertainty = NUMBER_OF_CLASSES / strength
    return probability, uncertainty


def validate_native_reconstruction(table, scales, label):
    three_mt, tmc = calibrated_branch_evidence(table, scales, label)
    reconstructed_p, reconstructed_u = mix_calibrated_evidence(
        three_mt,
        tmc,
        table["W_3MT"].to_numpy(dtype=float),
    )
    probability_error = float(np.max(np.abs(
        reconstructed_p - table["FINAL_P_pMCI"].to_numpy(dtype=float)
    )))
    uncertainty_error = float(np.max(np.abs(
        reconstructed_u - table["FINAL_UNCERTAINTY"].to_numpy(dtype=float)
    )))
    if probability_error > RECONSTRUCTION_TOLERANCE:
        raise ValueError(
            f"{label}: probability reconstruction error {probability_error} "
            f"exceeds {RECONSTRUCTION_TOLERANCE}."
        )
    if uncertainty_error > RECONSTRUCTION_TOLERANCE:
        raise ValueError(
            f"{label}: uncertainty reconstruction error {uncertainty_error} "
            f"exceeds {RECONSTRUCTION_TOLERANCE}."
        )
    return three_mt, tmc, probability_error, uncertainty_error


runs = {}
inventory_rows = []

for seed in SEEDS:
    for fold in OUTER_FOLDS:
        learned_root = learned_run_root(seed, fold)
        fixed_root = fixed_run_root(seed, fold)
        learned_paths = run_paths(learned_root)
        fixed_paths = run_paths(fixed_root)

        for experiment_label, paths in [
            ("learned", learned_paths),
            ("fixed", fixed_paths),
        ]:
            missing = [name for name, path in paths.items() if not path.is_file()]
            if missing:
                raise FileNotFoundError(
                    f"{experiment_label} seed {seed}, fold {fold} missing {missing}."
                )

        learned_configuration = load_json(learned_paths["configuration"])
        fixed_configuration = load_json(fixed_paths["configuration"])
        learned_success = load_json(learned_paths["success"])
        fixed_success = load_json(fixed_paths["success"])

        explicit_identity_checks = {
            "learned configuration experiment": (
                learned_configuration.get("experiment_name")
                == LEARNED_EXPERIMENT_NAME
            ),
            "fixed configuration experiment": (
                fixed_configuration.get("experiment_name")
                == FIXED_EXPERIMENT_NAME
            ),
            "learned configuration task": (
                learned_configuration.get("task") == TASK_NAME
            ),
            "fixed configuration task": (
                fixed_configuration.get("task") == TASK_NAME
            ),
            "learned configuration seed": (
                int(learned_configuration.get("random_seed", -1)) == seed
            ),
            "fixed configuration seed": (
                int(fixed_configuration.get("random_seed", -1)) == seed
            ),
            "learned configuration fold": (
                int(learned_configuration.get("fold", -1)) == fold
            ),
            "fixed configuration fold": (
                int(fixed_configuration.get("fold", -1)) == fold
            ),
            "learned completion experiment": (
                learned_success.get("experiment_name")
                == LEARNED_EXPERIMENT_NAME
            ),
            "fixed completion experiment": (
                fixed_success.get("experiment_name")
                == FIXED_EXPERIMENT_NAME
            ),
        }
        failed_explicit_identity = [
            name for name, passed in explicit_identity_checks.items() if not passed
        ]
        if failed_explicit_identity:
            raise ValueError(
                f"Seed {seed}, fold {fold} failed explicit identity checks: "
                f"{failed_explicit_identity}"
            )

        learned_identity = learned_configuration.get("run_identity")
        fixed_identity = fixed_configuration.get("run_identity")
        if not isinstance(learned_identity, dict) or not isinstance(fixed_identity, dict):
            raise ValueError(f"Missing run identity for seed {seed}, fold {fold}.")

        mismatched_invariants = [
            key for key in TOP_LEVEL_IDENTITY_INVARIANTS
            if learned_identity.get(key) != fixed_identity.get(key)
        ]
        if mismatched_invariants:
            raise ValueError(
                f"Seed {seed}, fold {fold} identity mismatch: {mismatched_invariants}"
            )

        learned_architecture = dict(
            learned_identity.get("architecture_and_training", {})
        )
        fixed_architecture = dict(
            fixed_identity.get("architecture_and_training", {})
        )
        for key in EXPECTED_GATE_ABLATION_DIFFERENCES:
            learned_architecture.pop(key, None)
            fixed_architecture.pop(key, None)
        if learned_architecture != fixed_architecture:
            differing = sorted(
                key for key in set(learned_architecture).intersection(fixed_architecture)
                if learned_architecture[key] != fixed_architecture[key]
            )
            raise ValueError(
                f"Seed {seed}, fold {fold} differs outside gate ablation: {differing}"
            )

        completion_checks = {
            "learned status": learned_success.get("status") == "complete",
            "fixed status": fixed_success.get("status") == "complete",
            "learned identity": learned_success.get("run_identity") == learned_identity,
            "fixed identity": fixed_success.get("run_identity") == fixed_identity,
            "learned prediction hash": (
                learned_success.get("test_predictions_sha256")
                == sha256(learned_paths["test"])
            ),
            "fixed prediction hash": (
                fixed_success.get("test_predictions_sha256")
                == sha256(fixed_paths["test"])
            ),
            "learned metrics hash": (
                learned_success.get("test_metrics_sha256")
                == sha256(learned_paths["metrics"])
            ),
            "fixed metrics hash": (
                fixed_success.get("test_metrics_sha256")
                == sha256(fixed_paths["metrics"])
            ),
        }
        failed = [name for name, passed in completion_checks.items() if not passed]
        if failed:
            raise ValueError(
                f"Seed {seed}, fold {fold} failed completion checks: {failed}"
            )

        learned_validation = pd.read_csv(learned_paths["validation"])
        learned_test = pd.read_csv(learned_paths["test"])
        fixed_validation = pd.read_csv(fixed_paths["validation"])
        fixed_test = pd.read_csv(fixed_paths["test"])

        for split_label, table in [
            ("learned validation", learned_validation),
            ("learned test", learned_test),
            ("fixed validation", fixed_validation),
            ("fixed test", fixed_test),
        ]:
            validate_prediction_table(
                table,
                f"{split_label}, seed {seed}, fold {fold}",
            )

        for split_name, learned_table, fixed_table in [
            ("validation", learned_validation, fixed_validation),
            ("test", learned_test, fixed_test),
        ]:
            learned_key = (
                learned_table[["RID", "TARGET"]]
                .sort_values("RID").reset_index(drop=True)
            )
            fixed_key = (
                fixed_table[["RID", "TARGET"]]
                .sort_values("RID").reset_index(drop=True)
            )
            if not learned_key.equals(fixed_key):
                raise ValueError(
                    f"Seed {seed}, fold {fold} {split_name} participants differ."
                )

        if set(learned_validation["RID"]).intersection(set(learned_test["RID"])):
            raise ValueError(f"Seed {seed}, fold {fold}: validation/test RID overlap.")
        if not np.allclose(fixed_validation["W_3MT"], 0.5, atol=1e-8):
            raise ValueError(f"Fixed validation weights are not 0.5: {seed}/{fold}")
        if not np.allclose(fixed_test["W_3MT"], 0.5, atol=1e-8):
            raise ValueError(f"Fixed test weights are not 0.5: {seed}/{fold}")

        learned_scales = load_checkpoint_scales(learned_paths["best_checkpoint"])
        fixed_scales = load_checkpoint_scales(fixed_paths["best_checkpoint"])

        learned_val_three, learned_val_tmc, learned_val_pe, learned_val_ue = (
            validate_native_reconstruction(
                learned_validation,
                learned_scales,
                f"learned validation seed {seed}, fold {fold}",
            )
        )
        learned_test_three, learned_test_tmc, learned_test_pe, learned_test_ue = (
            validate_native_reconstruction(
                learned_test,
                learned_scales,
                f"learned test seed {seed}, fold {fold}",
            )
        )
        _, _, fixed_val_pe, fixed_val_ue = validate_native_reconstruction(
            fixed_validation,
            fixed_scales,
            f"fixed validation seed {seed}, fold {fold}",
        )
        _, _, fixed_test_pe, fixed_test_ue = validate_native_reconstruction(
            fixed_test,
            fixed_scales,
            f"fixed test seed {seed}, fold {fold}",
        )

        runs[(seed, fold)] = {
            "learned_configuration": learned_configuration,
            "fixed_configuration": fixed_configuration,
            "learned_validation": learned_validation,
            "learned_test": learned_test,
            "fixed_validation": fixed_validation,
            "fixed_test": fixed_test,
            "learned_scales": learned_scales,
            "fixed_scales": fixed_scales,
            "learned_val_three": learned_val_three,
            "learned_val_tmc": learned_val_tmc,
            "learned_test_three": learned_test_three,
            "learned_test_tmc": learned_test_tmc,
        }

        inventory_rows.append({
            "SEED": seed,
            "FOLD": fold,
            "VALIDATION_PARTICIPANTS": len(learned_validation),
            "TEST_PARTICIPANTS": len(learned_test),
            "LEARNED_3MT_SCALE": learned_scales["THREE_MT_SCALE"],
            "LEARNED_TMC_SCALE": learned_scales["TMC_SCALE"],
            "FIXED_3MT_SCALE": fixed_scales["THREE_MT_SCALE"],
            "FIXED_TMC_SCALE": fixed_scales["TMC_SCALE"],
            "MAX_LEARNED_VALIDATION_P_ERROR": learned_val_pe,
            "MAX_LEARNED_TEST_P_ERROR": learned_test_pe,
            "MAX_FIXED_VALIDATION_P_ERROR": fixed_val_pe,
            "MAX_FIXED_TEST_P_ERROR": fixed_test_pe,
            "MAX_LEARNED_VALIDATION_U_ERROR": learned_val_ue,
            "MAX_LEARNED_TEST_U_ERROR": learned_test_ue,
            "MAX_FIXED_VALIDATION_U_ERROR": fixed_val_ue,
            "MAX_FIXED_TEST_U_ERROR": fixed_test_ue,
            "INPUT_SHA256": learned_identity["input_sha256"],
            "VALIDATION_ROWS_SHA256": learned_identity["validation_rows_sha256"],
            "TEST_ROWS_SHA256": learned_identity["test_rows_sha256"],
            "MATCHED_IDENTITY": True,
        })


validated_run_inventory = pd.DataFrame(inventory_rows)
atomic_write_csv(
    validated_run_inventory,
    TABLE_DIR / "validated_run_and_reconstruction_inventory.csv",
)

print("Validated matched runs:", len(validated_run_inventory))
print("Maximum native probability reconstruction error:",
      validated_run_inventory.filter(like="_P_ERROR").to_numpy().max())
print("Maximum native uncertainty reconstruction error:",
      validated_run_inventory.filter(like="_U_ERROR").to_numpy().max())
display(validated_run_inventory.round(8))
print("\nAll 30 runs and all native evidence reconstructions passed.")


## 1.3. Select one constant weight from validation data and freeze it for test

The constant is tuned on the **learned model's own calibrated interaction pathway and modality-specific evidence**. This creates a same-model counterfactual: the trained branches and evidence scales are unchanged, and only the participant-specific gate is replaced.

The separately trained fixed-50/50 model is retained as an additional comparison, but it is not confused with this same-model counterfactual.


In [ ]:
# ============================================================
# 3. Validation-only weight selection and untouched-test application
# ============================================================

def safe_log_loss(targets, probabilities):
    clipped = np.clip(np.asarray(probabilities, dtype=float), 1e-7, 1.0 - 1e-7)
    return float(log_loss(targets, clipped, labels=[0, 1]))


def expected_calibration_error(targets, probabilities, bins=10):
    targets = np.asarray(targets, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    assignments = np.digitize(probabilities, edges[1:-1], right=True)
    value = 0.0
    for bin_index in range(bins):
        mask = assignments == bin_index
        if not np.any(mask):
            continue
        value += (
            np.mean(mask)
            * abs(float(np.mean(targets[mask])) - float(np.mean(probabilities[mask])))
        )
    return float(value)


def calculate_metrics(targets, probabilities):
    targets = np.asarray(targets, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(int)
    return {
        "ROC_AUC": float(roc_auc_score(targets, probabilities)),
        "AVERAGE_PRECISION": float(average_precision_score(targets, probabilities)),
        "ACCURACY": float(accuracy_score(targets, predictions)),
        "BALANCED_ACCURACY": float(balanced_accuracy_score(targets, predictions)),
        "BRIER_SCORE": float(brier_score_loss(targets, probabilities)),
        "NEGATIVE_LOG_LIKELIHOOD": safe_log_loss(targets, probabilities),
        "EXPECTED_CALIBRATION_ERROR": expected_calibration_error(
            targets, probabilities
        ),
    }


def select_validation_alpha(targets, three_mt_evidence, tmc_evidence):
    rows = []
    for alpha in ALPHA_GRID:
        probabilities, uncertainties = mix_calibrated_evidence(
            three_mt_evidence,
            tmc_evidence,
            float(alpha),
        )
        rows.append({
            "ALPHA_3MT": float(alpha),
            "VALIDATION_ROC_AUC": float(roc_auc_score(targets, probabilities)),
            "VALIDATION_NEGATIVE_LOG_LIKELIHOOD": safe_log_loss(
                targets, probabilities
            ),
            "VALIDATION_BRIER_SCORE": float(
                brier_score_loss(targets, probabilities)
            ),
            "VALIDATION_MEAN_UNCERTAINTY": float(np.mean(uncertainties)),
        })

    grid = pd.DataFrame(rows)
    maximum_auc = float(grid["VALIDATION_ROC_AUC"].max())
    candidates = grid.loc[
        np.isclose(grid["VALIDATION_ROC_AUC"], maximum_auc, atol=1e-12, rtol=0.0)
    ].copy()
    minimum_nll = float(candidates["VALIDATION_NEGATIVE_LOG_LIKELIHOOD"].min())
    candidates = candidates.loc[
        np.isclose(
            candidates["VALIDATION_NEGATIVE_LOG_LIKELIHOOD"],
            minimum_nll,
            atol=1e-12,
            rtol=0.0,
        )
    ].copy()
    candidates["DISTANCE_FROM_050"] = abs(candidates["ALPHA_3MT"] - 0.5)
    selected = candidates.sort_values(
        ["DISTANCE_FROM_050", "ALPHA_3MT"], ascending=[True, True]
    ).iloc[0]
    return float(selected["ALPHA_3MT"]), grid


CORE_METHOD_ORDER = [
    "Learned native gate",
    "Validation-tuned constant",
    "Learned branches fixed 0.50",
    "Learned raw TMC-only",
    "Learned raw 3MT-only",
    "Separately trained fixed 0.50",
]

FULL_METHOD_ORDER = CORE_METHOD_ORDER + [
    "Learned calibrated TMC endpoint",
    "Learned calibrated 3MT endpoint",
]


selected_weight_rows = []
validation_grid_rows = []
test_prediction_rows = []

for seed in SEEDS:
    for fold in OUTER_FOLDS:
        run = runs[(seed, fold)]
        validation = run["learned_validation"]
        test = run["learned_test"]
        fixed_test = run["fixed_test"]

        validation_targets = validation["TARGET"].to_numpy(dtype=int)
        selected_alpha, grid = select_validation_alpha(
            validation_targets,
            run["learned_val_three"],
            run["learned_val_tmc"],
        )
        grid.insert(0, "FOLD", fold)
        grid.insert(0, "SEED", seed)
        validation_grid_rows.append(grid)

        selected_row = grid.loc[
            np.isclose(grid["ALPHA_3MT"], selected_alpha, atol=1e-12)
        ].iloc[0]
        native_validation_metrics = calculate_metrics(
            validation_targets,
            validation["FINAL_P_pMCI"].to_numpy(dtype=float),
        )

        selected_weight_rows.append({
            "SEED": seed,
            "FOLD": fold,
            "VALIDATION_PARTICIPANTS": len(validation),
            "SELECTED_ALPHA_3MT": selected_alpha,
            "SELECTED_ALPHA_TMC": 1.0 - selected_alpha,
            "SELECTED_VALIDATION_ROC_AUC": float(
                selected_row["VALIDATION_ROC_AUC"]
            ),
            "NATIVE_GATE_VALIDATION_ROC_AUC": native_validation_metrics["ROC_AUC"],
            "SELECTED_MINUS_NATIVE_VALIDATION_ROC_AUC": float(
                selected_row["VALIDATION_ROC_AUC"]
                - native_validation_metrics["ROC_AUC"]
            ),
            "SELECTED_VALIDATION_NLL": float(
                selected_row["VALIDATION_NEGATIVE_LOG_LIKELIHOOD"]
            ),
            "NATIVE_GATE_MEAN_TEST_W_3MT": float(test["W_3MT"].mean()),
            "NATIVE_GATE_SD_TEST_W_3MT": float(test["W_3MT"].std(ddof=1)),
        })

        tuned_p, tuned_u = mix_calibrated_evidence(
            run["learned_test_three"],
            run["learned_test_tmc"],
            selected_alpha,
        )
        counterfactual_050_p, counterfactual_050_u = mix_calibrated_evidence(
            run["learned_test_three"],
            run["learned_test_tmc"],
            0.50,
        )
        calibrated_tmc_p, calibrated_tmc_u = mix_calibrated_evidence(
            run["learned_test_three"],
            run["learned_test_tmc"],
            0.0,
        )
        calibrated_three_p, calibrated_three_u = mix_calibrated_evidence(
            run["learned_test_three"],
            run["learned_test_tmc"],
            1.0,
        )

        fixed_lookup = fixed_test.set_index("RID")
        for row_index, row in test.reset_index(drop=True).iterrows():
            rid = int(row["RID"])
            target = int(row["TARGET"])
            fixed_row = fixed_lookup.loc[rid]
            methods = {
                "Learned native gate": (
                    float(row["FINAL_P_pMCI"]),
                    float(row["FINAL_UNCERTAINTY"]),
                ),
                "Validation-tuned constant": (
                    float(tuned_p[row_index]),
                    float(tuned_u[row_index]),
                ),
                "Learned branches fixed 0.50": (
                    float(counterfactual_050_p[row_index]),
                    float(counterfactual_050_u[row_index]),
                ),
                "Learned raw TMC-only": (
                    float(row["TMC_P_pMCI"]),
                    float(row["TMC_UNCERTAINTY"]),
                ),
                "Learned raw 3MT-only": (
                    float(row["THREE_MT_P_pMCI"]),
                    float(row["THREE_MT_UNCERTAINTY"]),
                ),
                "Separately trained fixed 0.50": (
                    float(fixed_row["FINAL_P_pMCI"]),
                    float(fixed_row["FINAL_UNCERTAINTY"]),
                ),
                "Learned calibrated TMC endpoint": (
                    float(calibrated_tmc_p[row_index]),
                    float(calibrated_tmc_u[row_index]),
                ),
                "Learned calibrated 3MT endpoint": (
                    float(calibrated_three_p[row_index]),
                    float(calibrated_three_u[row_index]),
                ),
            }
            for method, (probability, uncertainty) in methods.items():
                test_prediction_rows.append({
                    "SEED": seed,
                    "FOLD": fold,
                    "RID": rid,
                    "TARGET": target,
                    "METHOD": method,
                    "P_pMCI": probability,
                    "UNCERTAINTY": uncertainty,
                    "SELECTED_ALPHA_3MT": selected_alpha,
                })


selected_weights = pd.DataFrame(selected_weight_rows)
validation_weight_grid = pd.concat(validation_grid_rows, ignore_index=True)
test_predictions_long = pd.DataFrame(test_prediction_rows)

for seed in SEEDS:
    seed_core = test_predictions_long.loc[
        (test_predictions_long["SEED"] == seed)
        & (test_predictions_long["METHOD"] == "Learned native gate")
    ]
    if len(seed_core) != EXPECTED_PARTICIPANTS_PER_SEED:
        raise ValueError(f"Seed {seed} does not contain 544 pooled OOF participants.")
    if seed_core["RID"].nunique() != EXPECTED_PARTICIPANTS_PER_SEED:
        raise ValueError(f"Seed {seed} contains duplicate/missing OOF RIDs.")

atomic_write_csv(selected_weights, TABLE_DIR / "validation_selected_weights.csv")
atomic_write_csv(validation_weight_grid, TABLE_DIR / "validation_weight_grid.csv")
atomic_write_csv(
    test_predictions_long,
    PREDICTION_DIR / "all_methods_matched_test_predictions.csv",
)

print("Validation-selected weights (test labels were not used):")
display(selected_weights.round(6))
print("\nSelected 3MT weight summary:")
display(selected_weights["SELECTED_ALPHA_3MT"].describe().to_frame().round(6))


## 1.4. Compare test performance across complete 544-participant out-of-fold cohorts

Metrics are computed separately for each seed after combining its five untouched test folds. Across-seed mean ± sample SD describes training-run sensitivity. The same-model comparisons isolate the routing rule; the separately trained fixed model additionally reflects changed joint-training dynamics.


In [ ]:
# ============================================================
# 4. Seedwise pooled performance and matched across-seed differences
# ============================================================

performance_rows = []
for seed in SEEDS:
    for method in FULL_METHOD_ORDER:
        subset = test_predictions_long.loc[
            (test_predictions_long["SEED"] == seed)
            & (test_predictions_long["METHOD"] == method)
        ].sort_values(["FOLD", "RID"])
        metrics = calculate_metrics(subset["TARGET"], subset["P_pMCI"])
        performance_rows.append({
            "SEED": seed,
            "METHOD": method,
            "PARTICIPANTS": len(subset),
            **metrics,
        })

seedwise_performance = pd.DataFrame(performance_rows)
seedwise_performance["METHOD"] = pd.Categorical(
    seedwise_performance["METHOD"],
    categories=FULL_METHOD_ORDER,
    ordered=True,
)
seedwise_performance = seedwise_performance.sort_values(
    ["METHOD", "SEED"]
).reset_index(drop=True)

summary_rows = []
for method in FULL_METHOD_ORDER:
    method_table = seedwise_performance.loc[
        seedwise_performance["METHOD"] == method
    ]
    for metric in [
        "ROC_AUC", "AVERAGE_PRECISION", "BALANCED_ACCURACY",
        "BRIER_SCORE", "NEGATIVE_LOG_LIKELIHOOD",
        "EXPECTED_CALIBRATION_ERROR",
    ]:
        values = method_table[metric].to_numpy(dtype=float)
        summary_rows.append({
            "METHOD": method,
            "METRIC": metric,
            "N_SEEDS": len(values),
            "MEAN": float(np.mean(values)),
            "STANDARD_DEVIATION": float(np.std(values, ddof=1)),
            "MINIMUM": float(np.min(values)),
            "MAXIMUM": float(np.max(values)),
            "MEAN_PLUS_MINUS_SD": (
                f"{np.mean(values):.4f} ± {np.std(values, ddof=1):.4f}"
            ),
        })
across_seed_performance = pd.DataFrame(summary_rows)


PRIMARY_COMPARISONS = [
    ("Validation-tuned constant", "Learned native gate"),
    ("Learned branches fixed 0.50", "Learned native gate"),
    ("Validation-tuned constant", "Learned raw TMC-only"),
    ("Validation-tuned constant", "Separately trained fixed 0.50"),
]

difference_rows = []
for seed in SEEDS:
    indexed = seedwise_performance.loc[
        seedwise_performance["SEED"] == seed
    ].set_index("METHOD")
    for first, second in PRIMARY_COMPARISONS:
        difference_rows.append({
            "SEED": seed,
            "FIRST_METHOD": first,
            "SECOND_METHOD": second,
            "DIFFERENCE_DEFINITION": "first minus second; Brier uses second minus first",
            "ROC_AUC_DIFFERENCE": float(
                indexed.loc[first, "ROC_AUC"] - indexed.loc[second, "ROC_AUC"]
            ),
            "AP_DIFFERENCE": float(
                indexed.loc[first, "AVERAGE_PRECISION"]
                - indexed.loc[second, "AVERAGE_PRECISION"]
            ),
            "BALANCED_ACCURACY_DIFFERENCE": float(
                indexed.loc[first, "BALANCED_ACCURACY"]
                - indexed.loc[second, "BALANCED_ACCURACY"]
            ),
            "BRIER_IMPROVEMENT": float(
                indexed.loc[second, "BRIER_SCORE"]
                - indexed.loc[first, "BRIER_SCORE"]
            ),
        })

seedwise_differences = pd.DataFrame(difference_rows)
difference_summary_rows = []
for (first, second), group in seedwise_differences.groupby(
    ["FIRST_METHOD", "SECOND_METHOD"], sort=False
):
    for metric in [
        "ROC_AUC_DIFFERENCE", "AP_DIFFERENCE",
        "BALANCED_ACCURACY_DIFFERENCE", "BRIER_IMPROVEMENT",
    ]:
        values = group[metric].to_numpy(dtype=float)
        difference_summary_rows.append({
            "FIRST_METHOD": first,
            "SECOND_METHOD": second,
            "DIFFERENCE_METRIC": metric,
            "N_MATCHED_SEEDS": len(values),
            "MEAN_DIFFERENCE": float(np.mean(values)),
            "SD_DIFFERENCE": float(np.std(values, ddof=1)),
            "MINIMUM_DIFFERENCE": float(np.min(values)),
            "MAXIMUM_DIFFERENCE": float(np.max(values)),
            "FIRST_FAVOURED_IN_ALL_SEEDS": bool(np.all(values > 0.0)),
            "SECOND_FAVOURED_IN_ALL_SEEDS": bool(np.all(values < 0.0)),
        })
difference_across_seed_summary = pd.DataFrame(difference_summary_rows)

atomic_write_csv(seedwise_performance, TABLE_DIR / "seedwise_test_performance.csv")
atomic_write_csv(
    across_seed_performance,
    TABLE_DIR / "across_seed_test_performance.csv",
)
atomic_write_csv(
    seedwise_differences,
    TABLE_DIR / "seedwise_matched_method_differences.csv",
)
atomic_write_csv(
    difference_across_seed_summary,
    TABLE_DIR / "matched_difference_across_seed_summary.csv",
)

print("Seedwise pooled test performance:")
display(seedwise_performance.loc[
    seedwise_performance["METHOD"].isin(CORE_METHOD_ORDER)
].round(6))
print("\nAcross-seed performance:")
display(across_seed_performance.loc[
    across_seed_performance["METHOD"].isin(CORE_METHOD_ORDER)
].round(6))
print("\nMatched seedwise differences:")
display(seedwise_differences.round(6))


## 1.5. Paired participant bootstrap within each seed

Participants are resampled stratified by outcome, with identical bootstrap samples used for both methods. These intervals quantify participant-sampling uncertainty within each seed. The three seedwise point differences remain the evidence about training-run sensitivity and are not treated as 1,632 independent participants.


In [ ]:
# ============================================================
# 5. Stratified paired participant bootstrap within each seed
# ============================================================

def stratified_bootstrap_indices(targets, rng):
    targets = np.asarray(targets, dtype=int)
    pieces = []
    for outcome in (0, 1):
        indices = np.flatnonzero(targets == outcome)
        pieces.append(rng.choice(indices, size=len(indices), replace=True))
    combined = np.concatenate(pieces)
    rng.shuffle(combined)
    return combined


def metric_value(metric, targets, probabilities):
    if metric == "ROC_AUC":
        return float(roc_auc_score(targets, probabilities))
    if metric == "AVERAGE_PRECISION":
        return float(average_precision_score(targets, probabilities))
    if metric == "BALANCED_ACCURACY":
        predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(int)
        return float(balanced_accuracy_score(targets, predictions))
    if metric == "BRIER_SCORE":
        return float(brier_score_loss(targets, probabilities))
    raise KeyError(metric)


bootstrap_rows = []
for comparison_index, (first, second) in enumerate(PRIMARY_COMPARISONS):
    for seed in SEEDS:
        seed_table = test_predictions_long.loc[
            (test_predictions_long["SEED"] == seed)
            & (test_predictions_long["METHOD"].isin([first, second]))
        ].pivot_table(
            index=["FOLD", "RID", "TARGET"],
            columns="METHOD",
            values="P_pMCI",
            aggfunc="first",
        ).reset_index()
        if seed_table[[first, second]].isna().any().any():
            raise ValueError(f"Incomplete paired predictions: {seed}, {first}, {second}")

        targets = seed_table["TARGET"].to_numpy(dtype=int)
        first_p = seed_table[first].to_numpy(dtype=float)
        second_p = seed_table[second].to_numpy(dtype=float)
        rng = np.random.default_rng(
            BOOTSTRAP_BASE_SEED + 10000 * comparison_index + seed
        )

        for metric in [
            "ROC_AUC", "AVERAGE_PRECISION", "BALANCED_ACCURACY", "BRIER_SCORE"
        ]:
            first_point = metric_value(metric, targets, first_p)
            second_point = metric_value(metric, targets, second_p)
            if metric == "BRIER_SCORE":
                point_difference = second_point - first_point
                definition = "second minus first; positive favours first"
            else:
                point_difference = first_point - second_point
                definition = "first minus second; positive favours first"

            bootstrap_differences = np.empty(BOOTSTRAP_ITERATIONS, dtype=float)
            for iteration in range(BOOTSTRAP_ITERATIONS):
                indices = stratified_bootstrap_indices(targets, rng)
                first_value = metric_value(
                    metric, targets[indices], first_p[indices]
                )
                second_value = metric_value(
                    metric, targets[indices], second_p[indices]
                )
                if metric == "BRIER_SCORE":
                    bootstrap_differences[iteration] = second_value - first_value
                else:
                    bootstrap_differences[iteration] = first_value - second_value

            lower, upper = np.percentile(bootstrap_differences, [2.5, 97.5])
            descriptive_p = 2.0 * min(
                np.mean(bootstrap_differences <= 0.0),
                np.mean(bootstrap_differences >= 0.0),
            )
            bootstrap_rows.append({
                "SEED": seed,
                "FIRST_METHOD": first,
                "SECOND_METHOD": second,
                "METRIC": metric,
                "FIRST_POINT": first_point,
                "SECOND_POINT": second_point,
                "DIFFERENCE_DEFINITION": definition,
                "POINT_DIFFERENCE": point_difference,
                "DIFFERENCE_CI_95_LOWER": float(lower),
                "DIFFERENCE_CI_95_UPPER": float(upper),
                "CI_EXCLUDES_ZERO": bool(lower > 0.0 or upper < 0.0),
                "DESCRIPTIVE_TWO_SIDED_BOOTSTRAP_P": float(min(1.0, descriptive_p)),
                "BOOTSTRAP_ITERATIONS": BOOTSTRAP_ITERATIONS,
            })

within_seed_bootstrap = pd.DataFrame(bootstrap_rows)
atomic_write_csv(
    within_seed_bootstrap,
    TABLE_DIR / "within_seed_paired_participant_bootstrap.csv",
)

print("Paired participant bootstrap results:")
display(within_seed_bootstrap.round(6))


## 1.6. Diagnose routing, branch complementarity and modality-pattern behaviour

This section uses test labels **only for post-hoc diagnosis**, never for model or weight selection. It asks whether higher learned interaction pathway weight corresponds to lower interaction pathway participant-level loss, whether the branches disagree enough to create routing headroom, and whether weights respond to exact modality-availability patterns.

The oracle accuracy is an explicitly non-deployable upper-bound diagnostic: it marks a participant correct whenever either calibrated pathway is correct.


In [ ]:
# ============================================================
# 6. Gate alignment, branch disagreement, oracle ceiling, and patterns
# ============================================================

diagnostic_rows = []
diagnostic_participant_rows = []
gate_run_rows = []

for seed in SEEDS:
    seed_pieces = []
    for fold in OUTER_FOLDS:
        run = runs[(seed, fold)]
        test = run["learned_test"].reset_index(drop=True).copy()
        calibrated_three_p, _ = mix_calibrated_evidence(
            run["learned_test_three"], run["learned_test_tmc"], 1.0
        )
        calibrated_tmc_p, _ = mix_calibrated_evidence(
            run["learned_test_three"], run["learned_test_tmc"], 0.0
        )

        targets = test["TARGET"].to_numpy(dtype=int)
        epsilon = 1e-7
        three_clipped = np.clip(calibrated_three_p, epsilon, 1.0 - epsilon)
        tmc_clipped = np.clip(calibrated_tmc_p, epsilon, 1.0 - epsilon)
        three_loss = -(
            targets * np.log(three_clipped)
            + (1 - targets) * np.log(1.0 - three_clipped)
        )
        tmc_loss = -(
            targets * np.log(tmc_clipped)
            + (1 - targets) * np.log(1.0 - tmc_clipped)
        )
        three_better = three_loss < tmc_loss
        three_class = (calibrated_three_p >= CLASSIFICATION_THRESHOLD).astype(int)
        tmc_class = (calibrated_tmc_p >= CLASSIFICATION_THRESHOLD).astype(int)
        disagreement = three_class != tmc_class
        three_correct = three_class == targets
        tmc_correct = tmc_class == targets
        hard_gate_selects_three = test["W_3MT"].to_numpy(dtype=float) >= 0.5

        piece = pd.DataFrame({
            "SEED": seed,
            "FOLD": fold,
            "RID": test["RID"].to_numpy(dtype=int),
            "TARGET": targets,
            "W_3MT": test["W_3MT"].to_numpy(dtype=float),
            "W_TMC": test["W_TMC"].to_numpy(dtype=float),
            "ORIGINAL_MODALITY_COUNT": test["ORIGINAL_MODALITY_COUNT"].to_numpy(dtype=int),
            "CALIBRATED_3MT_P_pMCI": calibrated_three_p,
            "CALIBRATED_TMC_P_pMCI": calibrated_tmc_p,
            "THREE_MT_LOG_LOSS": three_loss,
            "TMC_LOG_LOSS": tmc_loss,
            "THREE_MT_RELATIVE_ADVANTAGE": tmc_loss - three_loss,
            "THREE_MT_BETTER": three_better,
            "PATHWAYS_DISAGREE": disagreement,
            "THREE_MT_CORRECT": three_correct,
            "TMC_CORRECT": tmc_correct,
            "HARD_GATE_SELECTS_3MT": hard_gate_selects_three,
        })
        piece["HARD_GATE_SELECTION_CORRECT"] = np.where(
            hard_gate_selects_three,
            three_correct,
            tmc_correct,
        )
        seed_pieces.append(piece)

        gate_run_rows.append({
            "SEED": seed,
            "FOLD": fold,
            "PARTICIPANTS": len(test),
            "MEAN_W_3MT": float(test["W_3MT"].mean()),
            "SD_W_3MT": float(test["W_3MT"].std(ddof=1)),
            "MIN_W_3MT": float(test["W_3MT"].min()),
            "MAX_W_3MT": float(test["W_3MT"].max()),
        })

    seed_table = pd.concat(seed_pieces, ignore_index=True)
    diagnostic_participant_rows.append(seed_table)
    correlation = spearmanr(
        seed_table["W_3MT"],
        seed_table["THREE_MT_RELATIVE_ADVANTAGE"],
        nan_policy="raise",
    )
    selector_target = seed_table["THREE_MT_BETTER"].astype(int)
    selector_auc = (
        float(roc_auc_score(selector_target, seed_table["W_3MT"]))
        if selector_target.nunique() == 2 else np.nan
    )
    disagreement_table = seed_table.loc[seed_table["PATHWAYS_DISAGREE"]]
    oracle_correct = seed_table["THREE_MT_CORRECT"] | seed_table["TMC_CORRECT"]

    diagnostic_rows.append({
        "SEED": seed,
        "PARTICIPANTS": len(seed_table),
        "MEAN_W_3MT": float(seed_table["W_3MT"].mean()),
        "SD_W_3MT": float(seed_table["W_3MT"].std(ddof=1)),
        "MEAN_W_3MT_WHEN_3MT_LOWER_LOSS": float(
            seed_table.loc[seed_table["THREE_MT_BETTER"], "W_3MT"].mean()
        ),
        "MEAN_W_3MT_WHEN_TMC_LOWER_OR_EQUAL_LOSS": float(
            seed_table.loc[~seed_table["THREE_MT_BETTER"], "W_3MT"].mean()
        ),
        "SPEARMAN_W3_VS_3MT_RELATIVE_ADVANTAGE": float(correlation.statistic),
        "SPEARMAN_P_VALUE_DESCRIPTIVE": float(correlation.pvalue),
        "W3_SELECTOR_ROC_AUC_FOR_3MT_LOWER_LOSS": selector_auc,
        "PATHWAY_CLASS_DISAGREEMENT_RATE": float(seed_table["PATHWAYS_DISAGREE"].mean()),
        "DISAGREEMENT_PARTICIPANTS": len(disagreement_table),
        "THREE_MT_CORRECT_WHEN_DISAGREE": float(
            disagreement_table["THREE_MT_CORRECT"].mean()
        ) if len(disagreement_table) else np.nan,
        "TMC_CORRECT_WHEN_DISAGREE": float(
            disagreement_table["TMC_CORRECT"].mean()
        ) if len(disagreement_table) else np.nan,
        "HARD_GATE_SELECTION_ACCURACY_WHEN_DISAGREE": float(
            disagreement_table["HARD_GATE_SELECTION_CORRECT"].mean()
        ) if len(disagreement_table) else np.nan,
        "CALIBRATED_3MT_ACCURACY": float(seed_table["THREE_MT_CORRECT"].mean()),
        "CALIBRATED_TMC_ACCURACY": float(seed_table["TMC_CORRECT"].mean()),
        "NONDEPLOYABLE_ORACLE_ACCURACY": float(oracle_correct.mean()),
        "ORACLE_HEADROOM_OVER_TMC": float(
            oracle_correct.mean() - seed_table["TMC_CORRECT"].mean()
        ),
    })


gate_participant_diagnostics = pd.concat(
    diagnostic_participant_rows, ignore_index=True
)
gate_routing_summary = pd.DataFrame(diagnostic_rows)
gate_run_summary = pd.DataFrame(gate_run_rows)


# Reconstruct the exact six-branch availability pattern from the prepared input
# path recorded in the matched run identity. Seed 42 is used only as a duplicate
# metadata reference; fold/RID/target assignments were already verified equal.
availability_rows = []
for fold in OUTER_FOLDS:
    configuration = runs[(42, fold)]["learned_configuration"]
    identity = configuration["run_identity"]
    input_path = Path(identity["input_path"])
    if not input_path.is_file():
        raise FileNotFoundError(f"Prepared input is missing: {input_path}")
    if sha256(input_path) != identity["input_sha256"]:
        raise ValueError(f"Prepared input hash changed: {input_path}")

    required_columns = [
        "RID", "MCI_PROGNOSIS_TARGET", "OUTER_FOLD", "DATA_ROLE",
        *BRANCH_MASK_COLUMNS,
    ]
    prepared = pd.read_csv(input_path, usecols=required_columns)
    test_rows = prepared.loc[
        (pd.to_numeric(prepared["OUTER_FOLD"], errors="raise").astype(int) == fold)
        & (prepared["DATA_ROLE"].astype(str).str.strip().str.lower() == "test")
    ].copy()
    test_rows["FOLD"] = fold
    availability_rows.append(test_rows)

availability = pd.concat(availability_rows, ignore_index=True)
availability["RID"] = pd.to_numeric(availability["RID"], errors="raise").astype(int)
availability["TARGET_PREPARED"] = pd.to_numeric(
    availability["MCI_PROGNOSIS_TARGET"], errors="raise"
).astype(int)
for column in BRANCH_MASK_COLUMNS:
    availability[column] = pd.to_numeric(
        availability[column], errors="raise"
    ).astype(int)
    if set(availability[column]) - {0, 1}:
        raise ValueError(f"{column} contains values outside 0/1.")

availability["MODALITY_PATTERN_BINARY"] = availability.apply(
    lambda row: "".join(str(int(row[column])) for column in BRANCH_MASK_COLUMNS),
    axis=1,
)
availability["MISSING_MODALITIES"] = availability.apply(
    lambda row: "None" if all(int(row[column]) == 1 for column in BRANCH_MASK_COLUMNS)
    else " + ".join(
        BRANCH_DISPLAY_NAMES[column]
        for column in BRANCH_MASK_COLUMNS
        if int(row[column]) == 0
    ),
    axis=1,
)
availability["RECONSTRUCTED_MODALITY_COUNT"] = availability[
    BRANCH_MASK_COLUMNS
].sum(axis=1).astype(int)

if availability[["FOLD", "RID"]].duplicated().any():
    raise ValueError("Prepared test availability contains duplicate fold/RID keys.")

gate_participant_diagnostics = gate_participant_diagnostics.merge(
    availability[[
        "FOLD", "RID", "TARGET_PREPARED", "MODALITY_PATTERN_BINARY",
        "MISSING_MODALITIES", "RECONSTRUCTED_MODALITY_COUNT",
    ]],
    on=["FOLD", "RID"],
    how="left",
    validate="many_to_one",
)
if gate_participant_diagnostics["MODALITY_PATTERN_BINARY"].isna().any():
    raise ValueError("At least one test prediction lacks a modality pattern.")
if not np.array_equal(
    gate_participant_diagnostics["TARGET"].to_numpy(dtype=int),
    gate_participant_diagnostics["TARGET_PREPARED"].to_numpy(dtype=int),
):
    raise ValueError("Prepared and prediction targets differ.")
if not np.array_equal(
    gate_participant_diagnostics["ORIGINAL_MODALITY_COUNT"].to_numpy(dtype=int),
    gate_participant_diagnostics["RECONSTRUCTED_MODALITY_COUNT"].to_numpy(dtype=int),
):
    raise ValueError("Prepared and prediction modality counts differ.")

gate_by_pattern = (
    gate_participant_diagnostics.groupby(
        ["MODALITY_PATTERN_BINARY", "MISSING_MODALITIES"], dropna=False
    )
    .agg(
        SEED_PARTICIPANT_OBSERVATIONS=("RID", "size"),
        UNIQUE_PARTICIPANTS=("RID", "nunique"),
        MEAN_W_3MT=("W_3MT", "mean"),
        SD_W_3MT=("W_3MT", "std"),
        MEAN_3MT_RELATIVE_ADVANTAGE=("THREE_MT_RELATIVE_ADVANTAGE", "mean"),
        PROPORTION_3MT_LOWER_LOSS=("THREE_MT_BETTER", "mean"),
        PATHWAY_CLASS_DISAGREEMENT_RATE=("PATHWAYS_DISAGREE", "mean"),
    )
    .reset_index()
    .sort_values(["UNIQUE_PARTICIPANTS", "MODALITY_PATTERN_BINARY"],
                 ascending=[False, True])
    .reset_index(drop=True)
)

gate_variation_decomposition = pd.DataFrame([{
    "POOLED_W3_STANDARD_DEVIATION": float(
        gate_participant_diagnostics["W_3MT"].std(ddof=1)
    ),
    "MEAN_WITHIN_RUN_W3_STANDARD_DEVIATION": float(
        gate_run_summary["SD_W_3MT"].mean()
    ),
    "SD_OF_RUN_MEAN_W3": float(gate_run_summary["MEAN_W_3MT"].std(ddof=1)),
    "MINIMUM_RUN_MEAN_W3": float(gate_run_summary["MEAN_W_3MT"].min()),
    "MAXIMUM_RUN_MEAN_W3": float(gate_run_summary["MEAN_W_3MT"].max()),
}])

atomic_write_csv(
    gate_participant_diagnostics,
    PREDICTION_DIR / "gate_participant_routing_diagnostics.csv",
)
atomic_write_csv(gate_routing_summary, TABLE_DIR / "gate_routing_summary_by_seed.csv")
atomic_write_csv(gate_run_summary, TABLE_DIR / "gate_weight_summary_by_seed_fold.csv")
atomic_write_csv(gate_by_pattern, TABLE_DIR / "gate_routing_by_modality_pattern.csv")
atomic_write_csv(
    gate_variation_decomposition,
    TABLE_DIR / "gate_weight_variation_decomposition.csv",
)

print("Gate routing diagnostics by seed:")
display(gate_routing_summary.round(6))
print("\nGate variation within versus between trained runs:")
display(gate_variation_decomposition.round(6))
print("\nGate behaviour by exact modality pattern:")
display(gate_by_pattern.round(6))


## 1.7. Thesis-ready summaries, figures and completion manifest

The compact table separates the native learned gate, the validation-tuned same-model constant, the same-model 50/50 counterfactual, the raw modality-specific evidence pathway, and the separately trained fixed model. No result is selected from the test set.


In [ ]:
# ============================================================
# 7. Compact tables, figures, and atomic completion marker
# ============================================================

thesis_rows = []
for method in CORE_METHOD_ORDER:
    row = {"METHOD": method}
    for metric in [
        "ROC_AUC", "AVERAGE_PRECISION", "BALANCED_ACCURACY",
        "BRIER_SCORE", "EXPECTED_CALIBRATION_ERROR",
    ]:
        match = across_seed_performance.loc[
            (across_seed_performance["METHOD"] == method)
            & (across_seed_performance["METRIC"] == metric)
        ]
        row[metric] = match.iloc[0]["MEAN_PLUS_MINUS_SD"]
    thesis_rows.append(row)
thesis_performance_table = pd.DataFrame(thesis_rows)

thesis_difference_table = difference_across_seed_summary.loc[
    difference_across_seed_summary["DIFFERENCE_METRIC"].isin([
        "ROC_AUC_DIFFERENCE", "AP_DIFFERENCE", "BRIER_IMPROVEMENT"
    ])
].copy()

atomic_write_csv(
    thesis_performance_table,
    TABLE_DIR / "thesis_tuned_constant_performance_table.csv",
)
atomic_write_csv(
    thesis_difference_table,
    TABLE_DIR / "thesis_tuned_constant_difference_table.csv",
)


# Selected-weight heatmap.
weight_matrix = selected_weights.pivot(
    index="SEED", columns="FOLD", values="SELECTED_ALPHA_3MT"
).reindex(index=SEEDS, columns=OUTER_FOLDS)
figure, axis = plt.subplots(figsize=(7.2, 3.2))
image = axis.imshow(weight_matrix.to_numpy(), vmin=0.0, vmax=1.0, cmap="Blues")
for row_index, seed in enumerate(SEEDS):
    for column_index, fold in enumerate(OUTER_FOLDS):
        value = float(weight_matrix.loc[seed, fold])
        axis.text(column_index, row_index, f"{value:.2f}",
                  ha="center", va="center",
                  color="white" if value > 0.55 else "black")
axis.set_xticks(range(len(OUTER_FOLDS)), labels=[str(fold) for fold in OUTER_FOLDS])
axis.set_yticks(range(len(SEEDS)), labels=[str(seed) for seed in SEEDS])
axis.set_xlabel("Outer fold")
axis.set_ylabel("Seed")
axis.set_title("Validation-selected constant 3MT evidence weight")
colorbar = figure.colorbar(image, ax=axis, fraction=0.045, pad=0.04)
colorbar.set_label("3MT weight")
figure.tight_layout()
selected_weight_figure_path = FIGURE_DIR / "validation_selected_weight_heatmap.png"
atomic_save_figure(figure, selected_weight_figure_path)
plt.show()


# Mean AUC with across-seed SD for the principal methods.
auc_table = across_seed_performance.loc[
    (across_seed_performance["METRIC"] == "ROC_AUC")
    & (across_seed_performance["METHOD"].isin(CORE_METHOD_ORDER))
].copy()
auc_table["METHOD"] = pd.Categorical(
    auc_table["METHOD"], categories=CORE_METHOD_ORDER, ordered=True
)
auc_table = auc_table.sort_values("METHOD")
figure, axis = plt.subplots(figsize=(9.2, 4.5))
positions = np.arange(len(auc_table))
axis.errorbar(
    positions,
    auc_table["MEAN"],
    yerr=auc_table["STANDARD_DEVIATION"],
    fmt="o",
    color="#19647E",
    ecolor="#7AA6B5",
    capsize=5,
    markersize=7,
)
axis.set_xticks(positions, labels=auc_table["METHOD"], rotation=22, ha="right")
axis.set_ylabel("ROC AUC")
axis.set_title("Controlled three-seed performance (mean ± SD)")
axis.grid(axis="y", alpha=0.25)
figure.tight_layout()
performance_figure_path = FIGURE_DIR / "controlled_method_auc_mean_sd.png"
atomic_save_figure(figure, performance_figure_path)
plt.show()


print("Thesis-ready performance table:")
display(thesis_performance_table)
print("\nThesis-ready matched differences:")
display(thesis_difference_table.round(6))
print("\nSelected-weight summary:")
display(selected_weights[[
    "SEED", "FOLD", "SELECTED_ALPHA_3MT", "SELECTED_ALPHA_TMC",
    "SELECTED_VALIDATION_ROC_AUC", "NATIVE_GATE_VALIDATION_ROC_AUC",
]].round(6))


output_files = sorted(
    [path for path in COMPARISON_ROOT.rglob("*") if path.is_file()]
)
output_hashes = {
    str(path.relative_to(COMPARISON_ROOT)): sha256(path)
    for path in output_files
    if path.name != "_SUCCESS.json"
}
completion_record = {
    "status": "complete",
    "analysis_name": COMPARISON_NAME,
    "task": TASK_NAME,
    "learned_experiment": LEARNED_EXPERIMENT_NAME,
    "fixed_experiment": FIXED_EXPERIMENT_NAME,
    "seeds": list(SEEDS),
    "outer_folds": list(OUTER_FOLDS),
    "alpha_grid": [float(value) for value in ALPHA_GRID],
    "selection_metric": PRIMARY_SELECTION_METRIC,
    "tie_breakers": [
        "lower validation negative log-likelihood",
        "closer to 0.50",
        "smaller 3MT weight",
    ],
    "test_labels_used_for_weight_selection": False,
    "retraining_performed": False,
    "validated_matched_runs": len(validated_run_inventory),
    "participants_per_seed": EXPECTED_PARTICIPANTS_PER_SEED,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "output_sha256": output_hashes,
    "completed_at": datetime.now().isoformat(timespec="seconds"),
}
atomic_write_json(completion_record, COMPARISON_ROOT / "_SUCCESS.json")

print("\n" + "=" * 72)
print("GATE-ROUTING AND VALIDATION-TUNED CONSTANT ANALYSIS COMPLETE")
print("=" * 72)
print("No training folders were modified.")
print("Completion marker:", COMPARISON_ROOT / "_SUCCESS.json")
print("Comparison folder:", COMPARISON_ROOT)
